# 4. Radio signals (without a dongle)

You do not need a USB radio to learn the idea. This notebook makes a pretend 1 kHz tone, plots its spectrum, then checks whether SoapySDR can see a real dongle.

*Using SoapySDR in a notebook of your own? Copy the `SOAPY_SDR_PLUGIN_PATH` line to the top of it.*

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

fs = 48_000
t = np.arange(fs) / fs
rng = np.random.default_rng(0)
tone = np.sin(2 * np.pi * 1000 * t) + 0.3 * rng.standard_normal(t.size)

freqs, power = signal.welch(tone, fs=fs, nperseg=2048)
fig, ax = plt.subplots(figsize=(6, 2.4))
ax.semilogy(freqs, power)
ax.set_xlim(0, 5000)
ax.set_title("Spectrum of a 1 kHz tone with noise")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("power")
plt.tight_layout()
plt.show()

os.environ.setdefault("SOAPY_SDR_PLUGIN_PATH", "/opt/conda/lib/SoapySDR/modules0.8")
try:
    import SoapySDR

    SoapySDR.setLogLevel(SoapySDR.SOAPY_SDR_FATAL)
    try:
        radios = SoapySDR.Device.enumerate()
    finally:
        SoapySDR.setLogLevel(SoapySDR.SOAPY_SDR_INFO)
    drivers = [m for m in SoapySDR.listModules() if SoapySDR.getModuleVersion(m)]
    if radios:
        for radio in radios:
            print("SoapySDR found a radio:", radio)
    elif not drivers:
        print("SoapySDR: its radio drivers did not load.")
        print("  Try Kernel → Restart Kernel…, then run this cell again.")
    else:
        print("SoapySDR: No radio plugged in — that's fine for now.")
except Exception as problem:
    print("SoapySDR could not look for radios:", problem)

try:
    from rtlsdr import RtlSdr

    print("pyrtlsdr is ready. With an RTL-SDR dongle plugged in, RtlSdr() would open it.")
except Exception as problem:
    print("pyrtlsdr is not available:", problem)